In [2]:
import pandas as pd
import numpy as np
import ast
import os 
os.chdir(r"C:\Users\stavz\Desktop\masters\APM")

# fasta = pd.read_csv("data/gencode.v49.transcripts.fa.gz", compression="gzip", header=None)

In [3]:
def read_fasta(path):
    records = []
    header = None
    seq_chunks = []

    import gzip
    opener = gzip.open if path.endswith(".gz") else open

    with opener(path, "rt") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_chunks)))
                header = line[1:]           # remove ">"
                seq_chunks = []
            else:
                seq_chunks.append(line)
        # last record
        if header is not None:
            records.append((header, "".join(seq_chunks)))
    
    return pd.DataFrame(records, columns=["header", "sequence"])

df = read_fasta("data/gencode.v49.transcripts.fa.gz")

h = df["header"].str.rstrip("|").str.split("|", expand=True)
h.columns = [
    "transcript_id",
    "gene_id",
    "otthumg_id",
    "otthumt_id",
    "transcript_name",
    "gene_name",
    "length",
    "type"
]

df = pd.concat([df, h], axis=1)



In [84]:
cols = ["seqname","source","feature","start","end","score","strand","frame","attribute"]

gtf = pd.read_csv(
    "annotations/gencode.v49.annotation.gtf.gz",
    sep="\t",
    comment="#",      # skip header/comments that start with '#'
    header=None,      # GTFs typically have no header row
    names=cols,
    dtype=str,        # keep as strings; convert later if needed
    engine="python",
    na_filter=False
)

In [ ]:
#transcript filtration

import re
import pandas as pd
genes = pd.read_csv("data/primary_genes_all_features.csv")
# genes = genes[genes["is_mane_select"] == True]
# genes = genes[genes["feature"].isin(["start_codon", "stop_codon", "CDS", "transcript", "exon", "UTR"])]
gtf_filtered = gtf[
    gtf["attribute"].str.contains("basic|GENCODE_Primary|MANE_Select", regex=True)
    & (gtf["feature"] == "transcript")
]


attr = gtf_filtered["attribute"].fillna("")

# 1) Transcript ID
gtf_filtered["transcript_id"] = attr.str.extract(r'transcript_id\s+"([^"]+)"')

# 2) All tags (collect every occurrence)
gtf_filtered["tags"] = attr.apply(lambda s: re.findall(r'tag\s+"([^"]+)"', s))

# 3) (Optional) also keep a lowercase version for easy searching
gtf_filtered["tags_lower"] = gtf_filtered["tags"].apply(lambda L: [t.lower() for t in L])

# 4) Quick checks
has_mane_case_exact = gtf_filtered["tags"].apply(lambda L: "MANE_Select" in L)
has_mane_case_ins   = gtf_filtered["tags_lower"].apply(lambda L: "mane_select" in L)

print("Exact-case MANE_Select count:", has_mane_case_exact.sum())
print("Case-insensitive MANE_Select count:", has_mane_case_ins.sum())

ids = gtf_filtered["transcript_id"].unique()
genes = genes[genes["transcript_id"].isin(ids)]
genes["tags"] = genes["transcript_id"].map(gtf_filtered.set_index("transcript_id")["tags"])




C:\Users\stavz\AppData\Local\Temp\ipykernel_15640\392511463.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtf_filtered["transcript_id"] = attr.str.extract(r'transcript_id\s+"([^"]+)"')
C:\Users\stavz\AppData\Local\Temp\ipykernel_15640\392511463.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtf_filtered["tags"] = attr.apply(lambda s: re.findall(r'tag\s+"([^"]+)"', s))
C:\Users\stavz\AppData\Local\Temp\ipykernel_15640\392511463.py:22: SettingWithCopyWarning: 
A value is trying to be set on a cop

Exact-case MANE_Select count: 19276
Case-insensitive MANE_Select count: 19276


In [160]:
import pandas as pd

def build_tx_map(exons, strand):
    tx_map, tpos = [], 1
    for _, row in exons.iterrows():
        length = row["end"] - row["start"] + 1
        tx_map.append((row["start"], row["end"], tpos, tpos + length - 1))  # (g_start, g_end, t_start, t_end) 1-based
        tpos += length
    return tx_map

def genomic_to_tx(gen_pos, tx_map, strand):
    for g_start, g_end, t_start, t_end in tx_map:
        if g_start <= gen_pos <= g_end:
            if strand == "+":
                # left→right within exon
                return t_start + (gen_pos - g_start)
            else:
                # right→left within exon
                return t_end - (gen_pos - g_start)
    return None


def extract_transcript_regions(gtf_df, fasta_df, transcript_id, gene_name, tags, include_stop_in_cds=False):
    """
    Returns:
        full_seq, five_utr_seq, cds_seq, three_utr_seq, meta
    Assumptions:
        - FASTA is a GENCODE transcripts.fa (already 5'->3' transcript orientation).
        - GTF features are genomic coordinates (exon/CDS/stop_codon present).
    """

    # ---- 1) transcript sequence from FASTA ----
    seq_row = fasta_df.loc[fasta_df["transcript_id"] == transcript_id]
    if seq_row.empty:
        raise ValueError(f"Transcript {transcript_id} not in FASTA dataframe")
    full_seq = seq_row.iloc[0]["sequence"]
    L = len(full_seq)

    # ---- 2) GTF rows for this transcript ----
    df = gtf_df[gtf_df["transcript_id"] == transcript_id].copy()
    if df.empty:
        raise ValueError(f"Transcript {transcript_id} not in GTF dataframe")
    strand = df["strand"].iloc[0]

    # ---- 3) Exons in transcript order ----
    exons = df[df["feature"] == "exon"].copy()
    exons = exons.sort_values(by="start", ascending=(strand == "+"))

    # ---- 4) Genomic→transcript map ----
    tx_map = build_tx_map(exons, strand)

    # ---- 5) CDS genomic bounds ----
    cds_rows = df[df["feature"] == "CDS"]
    if cds_rows.empty:
        # Noncoding: 5' UTR = full, no CDS, no 3' UTR
        return full_seq, full_seq, "", "", {"strand": strand, "cds": None, "stop": None}

    cds_start_gen = cds_rows["start"].min()
    cds_end_gen   = cds_rows["end"].max()

    # Map CDS ends to transcript coords
    cds_start_tx = genomic_to_tx(cds_start_gen, tx_map, strand)
    cds_end_tx   = genomic_to_tx(cds_end_gen,   tx_map, strand)
    if cds_start_tx is None or cds_end_tx is None:
        raise ValueError("CDS boundaries could not be mapped to transcript coordinates")

    # Normalize order (handles '-' strand naturally)
    cds_start_tx, cds_end_tx = sorted([cds_start_tx, cds_end_tx])

    # ---- 6) Optional: refine with stop_codon ----
    stop_rows = df[df["feature"] == "stop_codon"]
    stop_meta = None
    utr3_start_tx = cds_end_tx + 1  # default: UTR starts right after CDS features

    if not stop_rows.empty:
        # For safety, use the outer stop span (handles 3-nt or split features)
        stop_start_gen = stop_rows["start"].min()
        stop_end_gen   = stop_rows["end"].max()

        stop_start_tx = genomic_to_tx(stop_start_gen, tx_map, strand)
        stop_end_tx   = genomic_to_tx(stop_end_gen,   tx_map, strand)


        if (stop_start_tx is not None) and (stop_end_tx is not None):
            stop_left_tx, stop_right_tx = sorted([stop_start_tx, stop_end_tx])
            stop_meta = {"stop_start_tx": stop_left_tx, "stop_end_tx": stop_right_tx}

            if include_stop_in_cds:
                # Extend CDS to include the stop triplet
                cds_end_tx = max(cds_end_tx, stop_right_tx)
                utr3_start_tx = cds_end_tx + 1
            else:
                # Start 3'UTR immediately AFTER the stop codon (not just after last CDS base)
                utr3_start_tx = stop_right_tx + 1

    # ---- 7) Slice regions (1-based -> 0-based) ----
    # Guard against edge cases (e.g., no UTR)
    cds_start_tx = max(1, min(cds_start_tx, L))
    cds_end_tx   = max(1, min(cds_end_tx,   L))
    utr3_start_tx = max(1, min(utr3_start_tx, L+1))  # +1 allows empty 3' UTR if CDS ends at L

    five_utr_seq  = full_seq[:cds_start_tx-1]             # [1 .. cds_start_tx-1]
    cds_seq       = full_seq[cds_start_tx-1:cds_end_tx]    # [cds_start_tx .. cds_end_tx]
    three_utr_seq = full_seq[utr3_start_tx-1:]             # [utr3_start_tx .. L]

    # ---- 8) Meta & sanity checks ----
    meta = {
        "strand": strand,
        "tx_length": L,
        "cds_start_tx": cds_start_tx,
        "cds_end_tx": cds_end_tx,
        "utr3_start_tx": utr3_start_tx,
        "len_utr5": len(five_utr_seq),
        "len_cds": len(cds_seq),
        "len_utr3": len(three_utr_seq),
        "stop": stop_meta,
        "include_stop_in_cds": include_stop_in_cds,
    }

    # # Optional sanity assertions (comment out if you prefer)
    assert 1 <= cds_start_tx <= cds_end_tx <= L, "CDS bounds out of transcript range"
    assert len(full_seq) == 3 + len(five_utr_seq) + len(cds_seq) + len(three_utr_seq), "Lengths don't sum to transcript"

    return gene_name, transcript_id, tags, full_seq, five_utr_seq, cds_seq, three_utr_seq, meta

# full_seq, five_utr, cds, three_utr, meta = extract_transcript_regions(genes, df, "ENST00000342505.5")
gene_name_lst,transcript_id_lst,tags_lst,full_seq_lst, five_utr_lst, cds_lst, three_utr_lst, meta_lst = [], [], [], [], [], [], [], []
for _,row in genes[genes["feature"] == "transcript"].iterrows():
    gene_name, transcript_id, tags, full_seq, five_utr, cds, three_utr, meta = extract_transcript_regions(genes, df, row["transcript_id"], row["gene_name"], row["tags"])
    gene_name_lst.append(gene_name)
    transcript_id_lst.append(transcript_id)
    tags_lst.append(tags)
    full_seq_lst.append(full_seq)
    five_utr_lst.append(five_utr)
    cds_lst.append(cds)
    three_utr_lst.append(three_utr)
    meta_lst.append(meta)
transcripts_df = pd.DataFrame({"gene_name": gene_name_lst, "transcript_id": transcript_id_lst, "tags": tags_lst, "full_seq": full_seq_lst, "five_utr": five_utr_lst, "cds": cds_lst, "three_utr": three_utr_lst, "meta": meta_lst})

In [162]:
transcripts_df.to_csv("data/transcripts_UTRs.csv", index=False)

In [168]:
transcripts_df["three_utr"].notna().sum()

np.int64(629)

In [163]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

records = [
    SeqRecord(Seq(row.three_utr), id=row.transcript_id, description="")
    for _, row in transcripts_df.iterrows() if row.three_utr
]
SeqIO.write(records, "data/fasta/three_utrs.fa", "fasta")


621

In [52]:
def get_miRNAs_targetscan(genes, top_n=200, weight_threshold=-0.2):
    targetscan_df = pd.read_csv(
        "data/miRNA/Predicted_Targets_Context_Scores.default_predictions.txt",
        sep="\t"
    )
    df = targetscan_df[(targetscan_df["Gene Tax ID"] == 9606) & (targetscan_df["Gene ID"].str.split('.').str[0].isin(genes["gene_id"].str.split('.').str[0]))]

    # aggregate per (gene, miRNA)
    gene_miRNA_best = (
        df.groupby(["Gene ID", "Gene Symbol", "miRNA"], as_index=False)
        .agg({
            "weighted context++ score": "min",                 # best score
            "UTR_start": lambda x: list(x),                    # list of starts
            "UTR end": lambda x: list(x),                      # list of ends
        })
    )

    # add number of binding sites
    gene_miRNA_best["num_sites"] = gene_miRNA_best["UTR_start"].apply(len)

    top_hits = (
        gene_miRNA_best
        .sort_values(["Gene ID","weighted context++ score"])
        .groupby("Gene ID")
        .head(top_n)       # top 3 miRNAs per gene
    )

    # Optional threshold for strong repression
    top_hits = top_hits[top_hits["weighted context++ score"] <= weight_threshold]
    return gene_miRNA_best, top_hits

top_n = 200
weight_threshold = -0.2
all_miRNAs, top_hits = get_miRNAs_targetscan(genes, top_n, weight_threshold)
all_miRNAs.to_csv("data/miRNA/all_miRNAs_targetscan.csv", index=False)
top_hits.to_csv(f"data/miRNA/top_{top_n}_threshold_{weight_threshold}_targetscan.csv", index=False)
